In [1]:
import gdown
import pandas as pd
gdown.download(r'https://bit.ly/3RhoNho', 'ns_202104.csv', quiet=False)
ns_df = pd.read_csv(r'ns_202104.csv', low_memory=False)
ns_df.head()

Downloading...
From: https://bit.ly/3RhoNho
To: c:\data\ns_202104.csv
100%|██████████| 57.6M/57.6M [00:10<00:00, 5.37MB/s]


,번호,도서명,저자,출판사,발행년도,ISBN,세트 ISBN,부가기호,권,주제분류번호,도서권수,대출건수,등록일자,Unnamed: 13
0,1,인공지능과 흙,김동훈 지음,민음사,2021,9788937444319,NaN,NaN,NaN,NaN,1,0,2021-03-19,NaN
1,2,가짜 행복 권하는 사회,김태형 지음,갈매나무,2021,9791190123969,NaN,NaN,NaN,NaN,1,0,2021-03-19,NaN
2,3,나도 한 문장 잘 쓰면 바랄 게 없겠네,김선영 지음,블랙피쉬,2021,9788968332982,NaN,NaN,NaN,NaN,1,0,2021-03-19,NaN
3,4,예루살렘 해변,"이도 게펜 지음, 임재희 옮김",문학세계사,2021,9788970759906,NaN,NaN,NaN,NaN,1,0,2021-03-19,NaN
4,5,김성곤의 중국한시기행 : 장강·황하 편,김성곤 지음,김영사,2021,9788934990833,NaN,NaN,NaN,NaN,1,0,2021-03-19,NaN


In [ ]:
import sqlite3

#db 객체 만들고 연결하기
conn = sqlite3.connect(r'C:\data\ns_lib.db')

#db 커서 객체(입력 및 조작 가능)만들기
c = conn.cursor()

#db 실행하고 테이블 만들기
c.execute("CREATE TABLE IF NOT EXISTS nslib_book \
    (name TEXT, author TEXT, borrow_counts INTEGER)")

# #db 삭제 명령문
# c.execute("DROB TABLE nslib_book")

In [ ]:
import pandas as pd

ns_df = pd.read_csv(r'ns_202104.csv', low_memory=False)
ns_df.head()


,번호,도서명,저자,출판사,발행년도,ISBN,세트 ISBN,부가기호,권,주제분류번호,도서권수,대출건수,등록일자,Unnamed: 13
0,1,인공지능과 흙,김동훈 지음,민음사,2021,9788937444319,NaN,NaN,NaN,NaN,1,0,2021-03-19,NaN
1,2,가짜 행복 권하는 사회,김태형 지음,갈매나무,2021,9791190123969,NaN,NaN,NaN,NaN,1,0,2021-03-19,NaN
2,3,나도 한 문장 잘 쓰면 바랄 게 없겠네,김선영 지음,블랙피쉬,2021,9788968332982,NaN,NaN,NaN,NaN,1,0,2021-03-19,NaN
3,4,예루살렘 해변,"이도 게펜 지음, 임재희 옮김",문학세계사,2021,9788970759906,NaN,NaN,NaN,NaN,1,0,2021-03-19,NaN
4,5,김성곤의 중국한시기행 : 장강·황하 편,김성곤 지음,김영사,2021,9788934990833,NaN,NaN,NaN,NaN,1,0,2021-03-19,NaN


In [ ]:
#판다스 데이터프레임은 for문으로 행을 가져오는 데에 최적화되어 있지 않기 때문에 시간이 많이 걸림
for index, row in ns_df.iterrows():
    c.execute("INSERT INTO nslib_book (name, author, borrow_counts) VALUES(?, ?, ?)", (row['도서명'], row['저자'], row['대출건수']))

In [39]:
#판다스 데이터프레임 sql 전달 메서드 : to_sql
book_df = ns_df[['도서명', '저자', '대출건수']]

#sql 테이블과 데이터프레임의 전달 열과 개수를 같게 만들어야 함!
book_df.columns = ['name', 'author', 'borrow_counts']

#to_sql 메서드로 판다스 데이터프레임의 데이터를 db에 sql로 전달하기
book_df.to_sql(r'nslib_book', conn, if_exists='replace', index=False) #직접 실행(cursor)가 아닌 연결하여 보내기이므로 conn(연결객체)!

401682

In [28]:
import sqlite3

#데이터프레임을 데이터베이스로 저장하기
conn = sqlite3.connect('ns_lib.db')
c = conn.cursor()
c.execute("CREATE TABLE IF NOT EXISTS nslib_book \
    (name TEXT, author TEXT, borrow_counts INTEGER)")
conn.commit()

book_df = ns_df[['도서명', '저자', '대출건수']]
book_df = book_df.rename(columns={'도서명':'name', '저자':'author', '대출건수': 'borrows_count'}) #열 이름과 열 개수는 맞춰줘야 함
book_df.to_sql(r'nslib_book', conn, if_exists='replace', index=False )

pd.read_sql_query("SELECT * FROM nslib_book", conn)



,name,author,borrows_count
0,인공지능과 흙,김동훈 지음,0
1,가짜 행복 권하는 사회,김태형 지음,0
2,나도 한 문장 잘 쓰면 바랄 게 없겠네,김선영 지음,0
3,예루살렘 해변,"이도 게펜 지음, 임재희 옮김",0
4,김성곤의 중국한시기행 : 장강·황하 편,김성곤 지음,0
...,...,...,...
401677,韓國現代詩大系,채만묵 編著,0
401678,뉴 웨이브,제임스 모나코 지음,0
401679,(최인훈 장편소설)화두,최인훈 지음,0
401680,독일 문학과 세계 문학,吳漢鎭 編著,0


In [31]:
#데이터베이스에 있는 데이터 데이터프레임으로 읽기
book_df = pd.read_sql_query('SELECT * FROM nslib_book', conn)

import sqlalchemy
book_df2 = pd.read_sql_table('nslib_book', r'sqlite:///ns_lib.db')
book_df2


,name,author,borrows_count
0,인공지능과 흙,김동훈 지음,0
1,가짜 행복 권하는 사회,김태형 지음,0
2,나도 한 문장 잘 쓰면 바랄 게 없겠네,김선영 지음,0
3,예루살렘 해변,"이도 게펜 지음, 임재희 옮김",0
4,김성곤의 중국한시기행 : 장강·황하 편,김성곤 지음,0
...,...,...,...
401677,韓國現代詩大系,채만묵 編著,0
401678,뉴 웨이브,제임스 모나코 지음,0
401679,(최인훈 장편소설)화두,최인훈 지음,0
401680,독일 문학과 세계 문학,吳漢鎭 編著,0


In [41]:
#DBMS로 통계량 구하기

#전체 행, 총 대출건수, 평균대출건수 구하기
ns_book = pd.read_sql_query('SELECT * FROM nslib_book', conn)
print(len(ns_book))

import numpy as np
a = np.sum(ns_book['borrows_count'])
b = np.mean(ns_book['borrows_count'])
print(a, b)


#DBMS로 구하기
#전체 행 개수
c.execute("SELECT COUNT(*) FROM nslib_book")
c.fetchone()
#총 대출건수
c.execute("SELECT SUM(borrows_count) FROM nslib_book")
c.fetchone()
#평균대출건수
c.execute("SELECT AVG(borrows_count) FROM nslib_book")
c.fetchone()

c.execute("SELECT COUNT(*), SUM(borrows_count), AVG(borrows_count) FROM nslib_book")
c.fetchone()

401682
4400145 10.95429966988812


(401682, 4400145, 10.95429966988812)

In [56]:
#데이터베이스 정렬하기

c.execute("SELECT * FROM nslib_book ORDER BY borrows_count")
c.fetchmany(10)

c.execute("SELECT * FROM nslib_book ORDER BY borrows_count DESC")
c.fetchmany(10)

c.execute("SELECT * FROM nslib_book ORDER BY borrows_count DESC LIMIT 20")
c.fetchall()

[('사피엔스 :유인원에서 사이보그까지, 인간 역사의 대담하고 위대한 질문 ', '유발 하라리 지음 ;조현욱 옮김', 1468),
 ('해커스 토익:Listening', 'David Cho 지음', 1065),
 ('7년의 밤 :정유정 장편소설 ', '정유정 저', 683),
 ('냉정과 열정사이:Blu', '츠지 히토나리 지음;양억관 옮김', 524),
 ('남한산성:김훈 장편소설', '김훈 지음', 501),
 ('해리포터와 혼혈왕자', '조앤 K. 롤링 지음;최인자 옮김', 451),
 ('해커스 토익:Listening', 'David Cho 지음', 440),
 ('다빈치 코드', '댄 브라운 지음;양선아 옮김', 440),
 ('신:베르나르 베르베르 장편소설', '베르나르 베르베르 지음;이세욱 옮김', 432),
 ('경제학 콘서트', '팀 하포드 지음;김명철 옮김', 425),
 ('엄마를 부탁해 :신경숙 장편소설 ', '신경숙 지음', 424),
 ('냉정과 열정사이:Rosso', '에쿠니 가오리 지음;김난주 옮김', 418),
 ('칼의 노래', '김훈 지음', 409),
 ('나미야 잡화점의 기적 :히가시노 게이고 장편소설 ', '히가시노 게이고 지음 ;양윤옥 옮김', 407),
 ('빅 픽처 :더글라스 케네디 장편소설 :진정 `나`를 위한 삶을 살고 싶었던 한 남자 이야기! ',
  '더글라스 케네디 지음 ;조동섭 옮김',
  398),
 ('11분:파울로 코엘료 장편소설', '파울로 코엘료 지음;이상해 옮김', 397),
 ('Word Smart', '애덤 로빈슨;프린스턴 리뷰팀 [같이]지음;NEXUS 사전편찬위원회 편역', 395),
 ('해리포터와 혼혈왕자', '조앤 K. 롤링 지음;최인자 옮김', 394),
 ('나마스테:박범신 장편소설', '박범신 지음', 394),
 ('1Q84 :무라카미 하루키 장편소설', '무라카미 하루키 지음 ;양윤옥 옮김', 390)]